In [ ]:
# ── Cell 1：内嵌 DataLoader 类定义 ───────────────────────────────────────────
import os, math, pandas as pd, numpy as np

_EVENT_PRIORITY = {'order': 0, 'trade': 1}

class DataLoader:
    def __init__(self, clean_dir):
        self.clean_dir = clean_dir
        self._df_level2 = self._df_order = self._df_trade = None
    def _lazy_load(self, attr, filename):
        if getattr(self, attr) is None:
            path = os.path.join(self.clean_dir, filename)
            df = pd.read_parquet(path, engine='pyarrow')
            setattr(self, attr, df)
            print(f'[DataLoader] {filename} 已加载：{len(df):,} 行')
        return getattr(self, attr)
    @property
    def level2(self): return self._lazy_load('_df_level2', 'level2.parquet')
    @property
    def order(self):  return self._lazy_load('_df_order',  'order.parquet')
    @property
    def trade(self):  return self._lazy_load('_df_trade',  'trade.parquet')
    def find_anchor(self, t_target):
        df  = self.level2
        idx = df.index.searchsorted(t_target, side='right') - 1
        if idx < 0: raise ValueError(f'找不到锚点：t_target={t_target} 早于最早时间 {df.index[0]}')
        return df.iloc[idx]
    def get_events(self, t_start, t_end):
        df_o = self.order
        df_order = df_o.loc[(df_o.index > t_start) & (df_o.index <= t_end)].copy()
        df_order['event_type'] = 'order'
        df_t = self.trade
        df_trade = df_t.loc[(df_t.index > t_start) & (df_t.index <= t_end)].copy()
        df_trade['event_type'] = 'trade'
        combined = pd.concat([df_order, df_trade], axis=0, sort=False)
        if combined.empty: return combined
        combined['_time']     = combined.index
        combined['_priority'] = combined['event_type'].map(_EVENT_PRIORITY)
        sort_keys = ['_time', '_seq', '_priority'] if 'seq_no' in combined.columns else ['_time', '_priority']
        if 'seq_no' in combined.columns: combined['_seq'] = combined['seq_no'].fillna(float('inf'))
        combined.sort_values(sort_keys, kind='stable', inplace=True)
        combined.set_index('_time', inplace=True)
        combined.index.name = 'time'
        drop_cols = ['_priority'] + (['_seq'] if '_seq' in combined.columns else [])
        combined.drop(columns=drop_cols, inplace=True)
        return combined

print('✓ DataLoader 类定义完成')

In [ ]:
# ── Cell 2：Mock 数据 & 测试工具 ──────────────────────────────────────────────
def make_mock_level2(n=10):
    times = [93000000 + i * 3000 for i in range(n)]
    data  = {'last': [2.950+i*0.001 for i in range(n)],
             'volume': [1000*(i+1) for i in range(n)],
             'amount': [2950.0*(i+1) for i in range(n)]}
    for j in range(1, 11):
        data[f'bp{j}'] = [round(2.950-(j-1)*0.001, 6)]*n; data[f'bv{j}'] = [1000]*n
        data[f'ap{j}'] = [round(2.951+(j-1)*0.001, 6)]*n; data[f'av{j}'] = [1000]*n
    return pd.DataFrame(data, index=pd.Index(times, name='time'))

def make_mock_order(n=20):
    times = [93000500 + i * 300 for i in range(n)]
    return pd.DataFrame({
        'side':       ['buy' if i%2==0 else 'sell' for i in range(n)],
        'price':      [2.948 if i%2==0 else 2.953 for i in range(n)],
        'quantity':   [200]*n, 'order_type': ['limit']*n,
        'symbol':     ['SH.510050']*n, 'date': [20240207]*n,
    }, index=pd.Index(times, name='time'))

def make_mock_trade(n=6):
    times = [93000600 + i * 500 for i in range(n)]
    return pd.DataFrame({
        'price': [2.951]*n, 'quantity': [100]*n,
        'symbol': ['SH.510050']*n, 'date': [20240207]*n,
    }, index=pd.Index(times, name='time'))

# 修改后：
def make_loader():
    dl = DataLoader('.') 
    dl._df_level2 = make_mock_level2()
    dl._df_order  = make_mock_order()
    dl._df_trade  = make_mock_trade()
    return dl

PASS, FAIL = '✓', '✗'
results = []
def check(name, condition):
    s = PASS if condition else FAIL
    results.append((s, name)); print(f'  {s}  {name}')

print('✓ Mock 数据和测试工具定义完成')

In [ ]:
# ── Cell 3：Data Quality Assertions ──────────────────────────────────────────
print('═'*50)
print('Data Quality Assertions')
print('═'*50)

loader = make_loader()

check('order.price 无 NaN',               loader.order['price'].notna().all())
check('order.quantity 无 NaN',            loader.order['quantity'].notna().all())
check('trade.price 无 NaN',               loader.trade['price'].notna().all())
check('trade.quantity 无 NaN',            loader.trade['quantity'].notna().all())
check('order.price > 0',                  (loader.order['price'] > 0).all())
check('order.quantity > 0',               (loader.order['quantity'] > 0).all())
check('trade.price > 0',                  (loader.trade['price'] > 0).all())
check('trade.quantity > 0',               (loader.trade['quantity'] > 0).all())
check('level2 时间戳单调非递减',           loader.level2.index.is_monotonic_increasing)
check('order 时间戳单调非递减',            loader.order.index.is_monotonic_increasing)
check('trade 时间戳单调非递减',            loader.trade.index.is_monotonic_increasing)
check('order.side 只含 buy/sell',          set(loader.order['side'].unique()).issubset({'buy','sell'}))
check('order.order_type 枚举合法',         set(loader.order['order_type'].unique()).issubset({'limit','market','stop'}))

In [ ]:
# ── Cell 4：find_anchor 测试 ──────────────────────────────────────────────────
print('═'*50)
print('find_anchor 测试')
print('═'*50)

loader = make_loader()

check('精确命中快照时间',                  loader.find_anchor(93000000).name == 93000000)
check('落在两快照之间，返回较早的',        loader.find_anchor(93001500).name == 93000000)
check('等于最后一行时间',                  loader.find_anchor(loader.level2.index[-1]).name == loader.level2.index[-1])
check('超过最后一行，返回最后一行',        loader.find_anchor(99999999).name == loader.level2.index[-1])

try:
    loader.find_anchor(90000000)
    check('早于第一行抛出 ValueError',     False)
except ValueError:
    check('早于第一行抛出 ValueError',     True)

In [ ]:
# ── Cell 5：get_events 测试 ───────────────────────────────────────────────────
print('═'*50)
print('get_events 测试')
print('═'*50)

loader = make_loader()
events = loader.get_events(93000000, 93010000)

check('返回 DataFrame',                    isinstance(events, pd.DataFrame))
check('t_start 不在结果中（左开）',        loader.order.index[0] not in loader.get_events(loader.order.index[0], loader.order.index[0]+5000).index)
check('t_end 包含在结果中（右闭）',        loader.order.index[2] in loader.get_events(93000000, loader.order.index[2]).index)
check('结果含 event_type 列',              'event_type' in events.columns)
check('event_type 只含 order/trade',       set(events['event_type'].unique()).issubset({'order','trade'}))
check('合并后时间戳单调非递减',            events.index.is_monotonic_increasing)
check('空区间返回空 DataFrame',            len(loader.get_events(80000000, 80001000)) == 0)
check('无临时列残留',                      all(c not in events.columns for c in ['_priority','_seq','_time']))

# 同时间戳 order 优先
ts = 93005000
loader2 = make_loader()
loader2._df_order = pd.DataFrame(
    {'side':['buy'],'price':[2.948],'quantity':[200],'order_type':['limit'],'symbol':['SH.510050'],'date':[20240207]},
    index=pd.Index([ts], name='time'))
loader2._df_trade = pd.DataFrame(
    {'price':[2.951],'quantity':[100],'symbol':['SH.510050'],'date':[20240207]},
    index=pd.Index([ts], name='time'))
ev2 = loader2.get_events(93000000, 93010000)
same = ev2[ev2.index == ts]
check('同时间戳 order 排在 trade 前',      same.iloc[0]['event_type'] == 'order' and same.iloc[1]['event_type'] == 'trade')

In [ ]:
# ── Cell 6：汇总结果 ──────────────────────────────────────────────────────────
total  = len(results)
passed = sum(1 for s, _ in results if s == PASS)
failed = total - passed
print('\n' + '═'*50)
print(f'  测试结果：{passed}/{total} 通过')
if failed:
    print(f'  ✗ 失败 {failed} 项：')
    for s, n in results:
        if s == FAIL: print(f'      ✗ {n}')
else:
    print('  🎉 全部通过！')
print('═'*50)